# 08 · Circulación de gran escala

¿El viento en altura, la convección amazónica y la MJO añaden algo a lo que ya saben la lluvia local, el suelo, el caudal y el mar?

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Es un análisis de investigación con un catálogo de fechas incompleto. Ante una emergencia llama al 115 (INDECI).

Las hipótesis, las cajas, los rezagos y el criterio de éxito se escribieron y subieron **antes** de descargar estas variables (`docs/08-circulacion.md`, sección 1):

| | Hipótesis | Variable | Rezagos |
|---|---|---|---|
| T16 | Viento del este en 200 hPa sobre el Altiplano distingue los días con huaico | `u200_altiplano` (20S-12S, 75W-65W) | 1 a 5 d |
| T17 | OLR bajo en la Amazonía y la fase de la MJO modulan | `olr_amazonia` (15S-5S, 70W-55W), RMM, OMI | 5 a 15 d |
| T18 | El SOI añade algo más allá del mar local | SOI del mes anterior | mes anterior |
| T19 | El flujo de humedad del este sobre la selva central precede | `u850/v850/q850_selva` (14S-8S, 72W-66W) | 1 a 5 d |
| T20 | Una vaguada o baja segregada en 500 hPa frente a Chile-Perú precede | `z500_chile_peru` (30S-15S, 85W-72W) y `z500_vaguada` | 0 a 4 d (≥ 1 en el modelo) |

Éxito: el AP fuera de muestra de `lluvia+antecedentes` sube más que el percentil 95 de lo que ganan las mismas columnas con las fechas desplazadas dentro de cada temporada, y se mantiene dejando episodios Niño fuera.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from illapa_research import circulacion as circ, modelo_diario as md, senales as S

N_PERM = 500  # permutaciones de la nula por (variable, rezago)
RAIZ = Path("..") if Path("../datos").exists() else Path(".")
D = md.cargar_datos(RAIZ)
Cir = circ.construir(RAIZ / "datos", D.F.index)
print(D.inventario["ultimo_dia"], "| columnas de circulación:", list(Cir.columns))
Cir.describe().T[["count", "mean", "std"]].round(2)

## 1. Rezagos 1 a 15 días: ¿qué distingue los días con huaico?

AUC de cada variable en cada rezago contra una nula que desplaza las etiquetas dentro de la temporada, con Benjamini-Hochberg (q = 0,10). `centrar=True` quita la media de cada temporada (queda solo el momento dentro de la temporada).

In [ ]:
F = pd.concat([D.F, Cir], axis=1)
mask = D.inventario["mask"]
cols = list(Cir.columns)
res = {}
for centrar in (False, True):
    r = S.rezagos_nula(F, D.y, cols, D.temporada, mask, lags=range(1, 16), centrar=centrar, n_null=N_PERM)
    res[centrar] = r
    print("centrado por temporada" if centrar else "sin centrar", "| pruebas:", r.attrs["n_pruebas"],
          "| pasan BH:", r.attrs["observados"], "| p<0,05:", r.attrs["observados_p05"],
          "| esperadas por azar:", round(r.attrs["esperados_p05_por_azar"], 1))
r = res[True]
r[r.senal].sort_values("p").head(25).round(3)

In [ ]:
(res[True].pivot(index="variable", columns="lag", values="auc") - 0.5).round(2)

## 2. Robustez de lo que pasa el filtro

In [ ]:
masc = S.mascaras_robustez(F, D.temporada, mask, D.temporadas)
rob = S.robustez(F, D.y, cols, D.temporada, masc, lags=range(1, 16), centrar=True, n_null=N_PERM)
print("pruebas:", rob.attrs["n_pruebas"], "| robustas:", int(rob["robusta"].sum()))
rob[rob["robusta"]].round(3)

## 3. ¿Mejoran el modelo diario? (el criterio de éxito)

Se agregan las columnas, desplazadas un día, a `lluvia+antecedentes` y se mide el AP fuera de muestra. La nula es la ganancia que dan las mismas columnas con las fechas desplazadas dentro de la temporada.

In [ ]:
N_NULA = 12  # 12 réplicas de etiquetas desplazadas por grupo (tarda ~15 min en total)
GRUPOS = {
    "T16 u200": [c for c in cols if c.startswith("u200")],
    "T17 olr + MJO": [c for c in cols if c.startswith(("olr", "rmm", "omi"))],
    "T18 SOI": ["soi_mes_previo"],
    "T19 viento y humedad 850": [c for c in cols if c.startswith(("u850", "v850", "q850", "flujo"))],
    "T20 z500": [c for c in cols if c.startswith("z500")],
    "todas": cols,
}
filas = []
for nombre, sub in GRUPOS.items():
    sub = [c for c in sub if c in Cir.columns]
    if not sub:
        continue
    g = circ.ganancia_ap(D, Cir, "temporada", n_jobs=8, solo=sub)
    nula = circ.ganancia_nula(D, Cir, n=N_NULA, n_jobs=8, solo=sub)
    ge = circ.ganancia_ap(D, Cir, "episodio", n_jobs=8, solo=sub)
    filas.append({"grupo": nombre, "columnas": g["columnas"], "AP sin": g["ap_sin"], "AP con": g["ap_con"],
                  "ganancia (temporada fuera)": g["ganancia"], "nula p95": np.percentile(nula, 95),
                  "nula media": nula.mean(), "ganancia (episodio fuera)": ge["ganancia"]})
tabla = pd.DataFrame(filas).set_index("grupo")
tabla["supera la nula"] = tabla["ganancia (temporada fuera)"] > tabla["nula p95"]
tabla.round(3)

## Lectura

Ver `docs/08-circulacion.md` para la interpretación con las cifras de la corrida, los datos usados y los límites. Las variables de gran escala vienen de reanálisis de malla gruesa (NCEP/NCAR R1, 2,5°): describen el patrón regional, no lo que pasa en una quebrada.